# Notebook 06 — Compliance checker evaluation
**Goal:** Measure how accurate `check_compliance()` actually is against real German health-advertising law, and quantify how much RAG grounding improved it.

By the end of this notebook you will have:
- Run the full compliance system against a 30-example hand-labeled test set
- Computed precision / recall / F1 / accuracy, overall and by category
- Isolated the RAG-grounded classifier vs. the old ungrounded classifier on the same claims (before/after comparison)
- Validated any fix against a holdout set never used to design it
- Saved everything to `data/eval/` and regenerated `data/eval/SUMMARY.md` — a single readable index of every run, so results never need to be dug out of individual JSON files

**Why this matters:** `check_compliance()` has two layers — a hardcoded phrase blocklist, and an LLM classifier. Until now nobody has measured whether the LLM layer is actually accurate, or just plausible-sounding. This notebook puts a number on it.

**Prerequisite:** Run notebook 05 first so the `eu-regulations` Pinecone namespace is populated — without it, the grounded classifier silently falls back to the ungrounded one and this eval measures the wrong thing.

**Quick reference:** for a fast summary of all past runs without re-running anything, just open `data/eval/SUMMARY.md`.

## Step 1 — Environment check

In [ ]:
import sys
sys.path.append('..')

from src.utils.config import OPENAI_API_KEY, PINECONE_API_KEY

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print('✅ Pinecone key loaded:', PINECONE_API_KEY[:8] + '...')

## Step 1b — Confirm the legal corpus is loaded
If this is empty, go run notebook 05 first — otherwise every "grounded" result below
will silently be the ungrounded fallback, and this eval won't measure what it claims to.

In [ ]:
from pinecone import Pinecone
from src.utils.config import PINECONE_INDEX_NAME
from src.ingestion.legal_docs import LEGAL_NAMESPACE

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
stats = index.describe_index_stats()
legal_count = stats.get('namespaces', {}).get(LEGAL_NAMESPACE, {}).get('vector_count', 0)

print(f'Legal provisions in \'{LEGAL_NAMESPACE}\': {legal_count}')
assert legal_count > 0, 'Legal corpus is empty — run notebook 05 first.'
print('✅ Legal corpus present, grounded classification will actually be grounded.')

## Step 2 — Load the labeled test set

In [ ]:
import json
from pathlib import Path

labeled_path = Path('../data/eval/compliance_labeled_set.json')
labeled_data = json.loads(labeled_path.read_text(encoding='utf-8'))
examples = labeled_data['examples']

print(f'Loaded {len(examples)} labeled examples')
from collections import Counter
print('By category:', dict(Counter(e['category'] for e in examples)))
print('By ground truth:', dict(Counter(e['ground_truth_compliant'] for e in examples)))

## Step 3 — Run the full system (`check_compliance`) on every example
This is what's actually deployed — blocklist first, RAG-grounded LLM as fallback.

In [ ]:
import time
from src.compliance.checker import check_compliance

full_system_results = []

for i, ex in enumerate(examples, 1):
    start = time.time()
    result = check_compliance(ex['claim'])
    latency_ms = round((time.time() - start) * 1000)

    predicted_compliant = result['compliant']
    correct = predicted_compliant == ex['ground_truth_compliant']

    full_system_results.append({
        'id': ex['id'],
        'claim': ex['claim'],
        'category': ex['category'],
        'ground_truth_compliant': ex['ground_truth_compliant'],
        'predicted_compliant': predicted_compliant,
        'correct': correct,
        'source': result['source'],
        'cited_sections': result.get('cited_sections', []),
        'reason': result.get('reason', ''),
        'latency_ms': latency_ms,
    })

    status = '✓' if correct else '✗'
    print(f'[{i}/{len(examples)}] {status} {ex["id"]} ({ex["category"]}) — predicted={predicted_compliant}, truth={ex["ground_truth_compliant"]}, source={result["source"]}')

print('\n✅ Full system eval complete.')

## Step 4 — Compute precision / recall / F1 / accuracy
"Non-compliant" is treated as the positive class — a false negative (missing a real
violation) is the worse failure mode for this system, so recall on violations matters most.

In [ ]:
def compute_metrics(results, label_key='ground_truth_compliant', pred_key='predicted_compliant'):
    """Binary classification metrics with 'non-compliant' (False) as the positive class."""
    tp = sum(1 for r in results if r[label_key] is False and r[pred_key] is False)
    fn = sum(1 for r in results if r[label_key] is False and r[pred_key] is True)
    fp = sum(1 for r in results if r[label_key] is True and r[pred_key] is False)
    tn = sum(1 for r in results if r[label_key] is True and r[pred_key] is True)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (tp + tn) / len(results) if results else 0.0

    return {
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3),
        'accuracy': round(accuracy, 3),
        'n': len(results),
    }


overall_metrics = compute_metrics(full_system_results)
print('=== FULL SYSTEM METRICS (blocklist + RAG-grounded LLM) ===')
for k, v in overall_metrics.items():
    print(f'  {k}: {v}')

## Step 5 — Break down accuracy by category
Shows which type of claim the system handles well vs. poorly.

In [ ]:
categories = sorted(set(r['category'] for r in full_system_results))
category_metrics = {}

print('=== ACCURACY BY CATEGORY ===')
for cat in categories:
    subset = [r for r in full_system_results if r['category'] == cat]
    correct = sum(1 for r in subset if r['correct'])
    acc = round(correct / len(subset), 3) if subset else 0.0
    category_metrics[cat] = {'n': len(subset), 'correct': correct, 'accuracy': acc}
    print(f'  {cat}: {correct}/{len(subset)} correct ({acc:.1%})')

## Step 6 — Before/after: ungrounded vs. RAG-grounded classifier
Isolates the layer-2 LLM decision on claims that dodge the blocklist — this is the
specific comparison that justifies the RAG-grounding work. Both classifiers see the
exact same claims; only the grounding differs.

In [ ]:
from src.compliance.checker import _llm_classify, _llm_classify_grounded, _BLOCKLIST_PATTERN

# Only test claims the blocklist doesn't already catch — that's where grounding matters.
non_blocklist_examples = [ex for ex in examples if not _BLOCKLIST_PATTERN.search(ex['claim'])]
print(f'{len(non_blocklist_examples)} of {len(examples)} examples reach layer 2 (not caught by blocklist)\n')

ungrounded_results = []
grounded_results = []

for i, ex in enumerate(non_blocklist_examples, 1):
    ungrounded = _llm_classify(ex['claim'])
    grounded = _llm_classify_grounded(ex['claim'])

    ungrounded_results.append({
        'id': ex['id'], 'category': ex['category'],
        'ground_truth_compliant': ex['ground_truth_compliant'],
        'predicted_compliant': ungrounded['compliant'],
    })
    grounded_results.append({
        'id': ex['id'], 'category': ex['category'],
        'ground_truth_compliant': ex['ground_truth_compliant'],
        'predicted_compliant': grounded['compliant'],
        'cited_sections': grounded.get('cited_sections', []),
    })

    match_u = '✓' if ungrounded['compliant'] == ex['ground_truth_compliant'] else '✗'
    match_g = '✓' if grounded['compliant'] == ex['ground_truth_compliant'] else '✗'
    print(f'[{i}/{len(non_blocklist_examples)}] {ex["id"]} — ungrounded {match_u} | grounded {match_g}')

print('\n✅ Before/after comparison complete.')

In [ ]:
ungrounded_metrics = compute_metrics(ungrounded_results)
grounded_metrics = compute_metrics(grounded_results)

print('=== LAYER-2 ONLY: UNGROUNDED (old) vs. RAG-GROUNDED (new) ===\n')
print(f'{"Metric":<12}{"Ungrounded":>12}{"Grounded":>12}{"Delta":>10}')
for key in ['accuracy', 'precision', 'recall', 'f1']:
    u, g = ungrounded_metrics[key], grounded_metrics[key]
    delta = round(g - u, 3)
    sign = '+' if delta >= 0 else ''
    print(f'{key:<12}{u:>12}{g:>12}{sign}{delta:>9}')

## Step 7 — Save results to `data/eval/compliance_results.json`
Persists everything so numbers survive notebook output clearing (we scrub outputs
before every commit to avoid leaking key prefixes) and can be diffed across reruns.

In [ ]:
from datetime import datetime, timezone

output = {
    'run_at': datetime.now(timezone.utc).isoformat(),
    'test_set_size': len(examples),
    'full_system': {
        'metrics': overall_metrics,
        'by_category': category_metrics,
        'predictions': full_system_results,
    },
    'layer2_comparison': {
        'n_non_blocklist_examples': len(non_blocklist_examples),
        'ungrounded_metrics': ungrounded_metrics,
        'grounded_metrics': grounded_metrics,
        'ungrounded_predictions': ungrounded_results,
        'grounded_predictions': grounded_results,
    },
}

# "Latest" — always overwritten, convenient for quick inspection.
results_path = Path('../data/eval/compliance_results.json')
results_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ Saved latest results to {results_path}')

# Archived — one file per run, never overwritten, so iterations are diffable
# against each other (e.g. "did precision improve after the last prompt change?").
runs_dir = Path('../data/eval/runs')
runs_dir.mkdir(parents=True, exist_ok=True)
run_label = output['run_at'].replace(':', '').replace('.', '')[:15]
archive_path = runs_dir / f'{run_label}.json'
archive_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ Archived this run to {archive_path}')

## Step 8 — Presentation-ready summary

In [ ]:
print('='*60)
print('COMPLIANCE CHECKER — EVALUATION SUMMARY')
print('='*60)
print(f'Test set: {len(examples)} hand-labeled claims ({dict(Counter(e["category"] for e in examples))})')
print()
print(f'Full system accuracy: {overall_metrics["accuracy"]:.1%}')
print(f'Full system F1 (non-compliant as positive): {overall_metrics["f1"]:.3f}')
print(f'  Precision: {overall_metrics["precision"]:.3f} | Recall: {overall_metrics["recall"]:.3f}')
print()
print('RAG grounding impact (layer 2 only, blocklist-dodging claims):')
print(f'  Ungrounded LLM accuracy: {ungrounded_metrics["accuracy"]:.1%}')
print(f'  RAG-grounded LLM accuracy: {grounded_metrics["accuracy"]:.1%}')
print(f'  Improvement: {(grounded_metrics["accuracy"] - ungrounded_metrics["accuracy"]):+.1%}')
print('='*60)

## Step 9 — Holdout set: is the improvement real, or overfit?

Any fix designed by looking directly at failing examples risks "solving the eval,"
not the underlying problem — re-testing on the same 30 examples can't tell them apart.

`compliance_holdout_set.json` was drafted independently, *after* the prompt fix,
covering the same category types with entirely new wording and scenarios. If accuracy
holds up here too, the fix generalizes. If it craters, the earlier improvement was
likely memorizing the original test set's specific phrasing.

In [ ]:
holdout_path = Path('../data/eval/compliance_holdout_set.json')
holdout_data = json.loads(holdout_path.read_text(encoding='utf-8'))
holdout_examples = holdout_data['examples']

print(f'Loaded {len(holdout_examples)} holdout examples')
print('By category:', dict(Counter(e['category'] for e in holdout_examples)))
print('By ground truth:', dict(Counter(e['ground_truth_compliant'] for e in holdout_examples)))

holdout_results = []
for i, ex in enumerate(holdout_examples, 1):
    start = time.time()
    result = check_compliance(ex['claim'])
    latency_ms = round((time.time() - start) * 1000)

    predicted_compliant = result['compliant']
    correct = predicted_compliant == ex['ground_truth_compliant']

    holdout_results.append({
        'id': ex['id'],
        'claim': ex['claim'],
        'category': ex['category'],
        'ground_truth_compliant': ex['ground_truth_compliant'],
        'predicted_compliant': predicted_compliant,
        'correct': correct,
        'source': result['source'],
        'cited_sections': result.get('cited_sections', []),
        'reason': result.get('reason', ''),
        'latency_ms': latency_ms,
    })

    status = '✓' if correct else '✗'
    print(f'[{i}/{len(holdout_examples)}] {status} {ex["id"]} ({ex["category"]}) — predicted={predicted_compliant}, truth={ex["ground_truth_compliant"]}')
    if not correct:
        print(f'    REASON: {result.get("reason", "")}')

holdout_metrics = compute_metrics(holdout_results)
print('\n=== HOLDOUT SET METRICS ===')
for k, v in holdout_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
holdout_output = {
    'run_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Holdout set — drafted after and independently of the prompt fix, never used to design it.',
    'test_set_size': len(holdout_examples),
    'metrics': holdout_metrics,
    'predictions': holdout_results,
}

holdout_results_path = Path('../data/eval/compliance_holdout_results.json')
holdout_results_path.write_text(json.dumps(holdout_output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ Saved latest holdout results to {holdout_results_path}')

holdout_run_label = holdout_output['run_at'].replace(':', '').replace('.', '')[:15]
holdout_archive_path = runs_dir / f'{holdout_run_label}_holdout.json'
holdout_archive_path.write_text(json.dumps(holdout_output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ Archived to {holdout_archive_path}')

print(f'\nMain set (fitted) accuracy: {overall_metrics["accuracy"]:.1%} | F1: {overall_metrics["f1"]:.3f}')
print(f'Holdout (unseen) accuracy:  {holdout_metrics["accuracy"]:.1%} | F1: {holdout_metrics["f1"]:.3f}')
gap = overall_metrics['accuracy'] - holdout_metrics['accuracy']
print(f'Gap: {gap:+.1%} — a small/negative gap suggests the fix generalizes rather than overfitting.')

## Step 10 — Regenerate the one-place summary

`data/eval/SUMMARY.md` is a single readable index of every archived run — accuracy/
precision/recall/F1 per run, plus the layer-2 comparison, without opening individual
JSON files. Regenerated from `data/eval/runs/` every time this cell runs, so it never
drifts out of sync with the archive.

In [ ]:
from src.utils.eval_summary import generate_eval_summary

summary_path = generate_eval_summary()
print(f'✅ Regenerated {summary_path}')
print(f'\n{summary_path.read_text(encoding="utf-8")}')

## Notes

**What this does and doesn't prove:** this measures binary compliant/non-compliant accuracy
against hand-labeled sets — it does *not* validate that cited section numbers are legally
precise (section-citation accuracy is a separate, harder problem we deliberately scoped out).
Treat the numbers as directional evidence, not a legal certification.

**Re-running:** Safe to re-run any time. `compliance_results.json` and
`compliance_holdout_results.json` always reflect the latest run; every run is also archived
under `data/eval/runs/<timestamp>[_holdout].json`, so you can diff precision/recall/F1 across
iterations after a prompt or blocklist change.

**Extending either test set:** Add examples to `compliance_labeled_set.json` (main/fitted set)
or `compliance_holdout_set.json` (generalization check), following the same `category` +
`rationale` convention. Keep them non-overlapping — the holdout set only means something if
its examples were never looked at while designing a fix.